<a href="https://colab.research.google.com/github/beyzadurdu6619/TrustLLM-Uncertainty-Quantification/blob/main/notebooks/06_week/temperature_scaling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. Set seed for reproducibility / Tekrarlanabilirlik için tohum belirleme
torch.manual_seed(42)
np.random.seed(42)

# 2. Generate synthetic overconfident logits and true labels
# Yapay olarak aşırı özgüvenli (yüksek değerli) logitler ve doğru etiketler üretme
num_samples = 1000
num_classes = 5

# High magnitude logits simulate an overconfident model
# Yüksek genlikli logitler modelin aşırı özgüvenli olmasını taklit eder
raw_logits = torch.randn(num_samples, num_classes) * 3.5
labels = torch.randint(0, num_classes, (num_samples,))

print(f"Generated Logits Shape: {raw_logits.shape}")
print(f"Generated Labels Shape: {labels.shape}")

Generated Logits Shape: torch.Size([1000, 5])
Generated Labels Shape: torch.Size([1000])


In [6]:
import torch.optim as optim


class TemperatureScaler(nn.Module):
    """Wrapper module to learn the optimal Temperature parameter (T) for logit scaling."""

    def __init__(self):
        super(TemperatureScaler, self).__init__()
        # Initialize T to 1.5 (must be > 0)
        # T değerini 1.5 olarak başlatıyoruz
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, logits):
        # Scale logits using the temperature parameter
        # Logitleri T parametresi ile ölçekle
        return self.scale_logits(logits)

    def scale_logits(self, logits):
        # Expand T to match the shape of the logits tensor
        # T tensorunu logit boyutuna genişlet
        temperature = self.temperature.unsqueeze(1).expand(
            logits.size(0), logits.size(1)
        )
        return logits / temperature

    def fit(self, logits, labels):
        """Optimizes the Temperature parameter (T) using Cross Entropy Loss on validation data."""
        criterion = nn.CrossEntropyLoss()

        # L-BFGS optimizer is standard for Temperature Scaling
        # L-BFGS optimizasyon algoritması bu işlem için standarttır
        optimizer = optim.LBFGS([self.temperature], lr=0.01, max_iter=50)

        def eval_loss():
            optimizer.zero_grad()
            loss = criterion(self.scale_logits(logits), labels)
            loss.backward()
            return loss

        optimizer.step(eval_loss)
        print(f"Optimal Temperature (T): {self.temperature.item():.4f}")


# Instantiate and train the scaler / Scaler'ı oluşturup eğitelim
scaler = TemperatureScaler()
scaler.fit(raw_logits, labels)

# Obtain calibrated logits / Kalibre edilmiş logitleri alalım
calibrated_logits = scaler(raw_logits)

Optimal Temperature (T): 1.8601


In [9]:
def compute_ece(logits, labels, n_bins=10):
    """Calculates Expected Calibration Error (ECE) for a given set of logits and labels."""
    # Convert logits to probabilities via Softmax
    # Logitleri Softmax ile olasılıklara dönüştür
    probs = F.softmax(logits, dim=1).detach().numpy()
    labels = labels.numpy()

    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    accuracies = predictions == labels

    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        bin_lower, bin_upper = bin_boundaries[i], bin_boundaries[i + 1]

        # Identify samples belonging to the current bin
        # Mevcut kovana düşen örnekleri filtrele
        in_bin = (confidences > bin_lower) & (confidences <= bin_upper)
        prop_in_bin = np.mean(in_bin)

        if prop_in_bin > 0:
            accuracy_in_bin = np.mean(accuracies[in_bin])
            avg_confidence_in_bin = np.mean(confidences[in_bin])
            # Accumulate weighted absolute difference
            # Ağırlıklı mutlak farkı ECE skoruna ekle
            ece += np.abs(accuracy_in_bin - avg_confidence_in_bin) * prop_in_bin

    return ece


# Evaluate ECE before and after calibration
# Kalibrasyon öncesi ve sonrası ECE skorlarını değerlendirelim
ece_before = compute_ece(raw_logits, labels)
ece_after = compute_ece(calibrated_logits, labels)

print(f"ECE Before Temperature Scaling: {ece_before:.4f}")
print(f"ECE After Temperature Scaling:  {ece_after:.4f}")

ECE Before Temperature Scaling: 0.5717
ECE After Temperature Scaling:  0.4211


In [10]:
class DropoutModel(nn.Module):
    """Simple Neural Network with Dropout for Monte Carlo Dropout evaluation."""

    def __init__(self, input_dim, num_classes):
        super(DropoutModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.dropout = nn.Dropout(p=0.3)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)


def enable_dropout(model):
    """Forces all Dropout layers in the model to remain active during evaluation."""
    for m in model.modules():
        if m.__class__.__name__.startswith("Dropout"):
            m.train()


def estimate_mc_uncertainty(model, x_input, num_samples=20):
    """Computes mean probabilities and epistemic uncertainty (variance) using MC Dropout."""
    model.eval()
    enable_dropout(model)  # Keep dropout active / Dropout'u açık tut

    mc_predictions = []

    with torch.no_grad():
        for _ in range(num_samples):
            logits = model(x_input)
            probs = F.softmax(logits, dim=-1)
            mc_predictions.append(probs)

    # Shape: (num_samples, batch_size, num_classes)
    mc_predictions = torch.stack(mc_predictions)

    # Mean prediction across MC passes / Ortamala olasılık tahmini
    mean_probs = torch.mean(mc_predictions, dim=0)

    # Variance across MC passes represents Epistemic Uncertainty
    # MC örnekleri arasındaki varyans Epistemic Belirsizliği verir
    epistemic_uncertainty = torch.var(mc_predictions, dim=0)

    return mean_probs, epistemic_uncertainty


# Test MC Dropout / MC Dropout test işlemi
sample_input = torch.randn(5, 10)  # 5 samples, 10 feature dimension
mc_model = DropoutModel(input_dim=10, num_classes=3)

mean_p, epistemic_u = estimate_mc_uncertainty(
    mc_model, sample_input, num_samples=15
)

print(f"Mean Probabilities Shape: {mean_p.shape}")
print(f"Epistemic Uncertainty Shape (Variance): {epistemic_u.shape}")

Mean Probabilities Shape: torch.Size([5, 3])
Epistemic Uncertainty Shape (Variance): torch.Size([5, 3])


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

class TemperatureScaler(nn.Module):
    def __init__(self):
        super(TemperatureScaler, self).__init__()
        # T parametresini başlangıçta 1.5 olarak tanımlıyoruz (Eğitilebilir bir parametre)
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, logits):
        # Logitleri T sayısına bölerek yumuşatıyoruz
        return logits / self.temperature

    def fit(self, valid_logits, valid_labels):
        # En uygun T değerini bulmak için L-BFGS optimizasyon algoritmasını kullanıyoruz
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.LBFGS([self.temperature], lr=0.01, max_iter=50)

        def eval_loss():
            optimizer.zero_grad()
            loss = criterion(self.forward(valid_logits), valid_labels)
            loss.backward()
            return loss

        optimizer.step(eval_loss)
        print(f"İdeal Sıcaklık Değeri (T): {self.temperature.item():.4f}")

In [5]:
def enable_dropout(model):
    """Test anında bile Dropout katmanlarını açık tutar."""
    for m in model.modules():
        if m.__class__.__name__.startswith("Dropout"):
            m.train()

def estimate_mc_uncertainty(model, x_input, num_samples=20):
    model.eval()
    enable_dropout(model) # Dropout'u zorla açık bırakıyoruz!

    mc_predictions = []

    # Aynı girdiyi modele 20 kez soruyoruz
    with torch.no_grad():
        for _ in range(num_samples):
            logits = model(x_input)
            probs = F.softmax(logits, dim=-1)
            mc_predictions.append(probs)

    # 20 tahminin listesi -> (20, batch_size, class_size)
    mc_predictions = torch.stack(mc_predictions)

    # Tahminlerin ortalaması (Nihai Tahmin)
    mean_probs = torch.mean(mc_predictions, dim=0)

    # Tahminler arasındaki VARYANS = Modelin Bilgisizlik Belirsizliği (Epistemic Uncertainty)
    epistemic_uncertainty = torch.var(mc_predictions, dim=0)

    return mean_probs, epistemic_uncertainty